# LLM agent in python

In [2]:
%pip install openai

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 20.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 24.5 MB/s  0:00:00

   ------ ---------------------------------  2/13 [sniffio]
   --------------- ------------------------  5/13 [idna]
   ------------------ ---------------------  6/13 [h11]
   ------------------------ ---------------  8/13 [pydantic]
   ------------------------ ---------------  8/13 [pydantic]
   ------------------------ ---------------  8/13 [pydantic]
   ------------------------ ---------------  8/13 [pydantic]
   ------------------------ ---------------  8/13 [pydantic]
   ------------------------ ---------------  8/13 [pydantic]
   ------------------------ ---------------  8/13 [pydantic]
   ------------------------ ---------------  8/13 [pydantic]
   --------------------------- ------------  9/13 [h

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import openai

print(openai.__version__)

3.28.0


### step1 One API call


##### open ai

In [ ]:
from openai import OpenAI

client = OpenAI(api_key = )

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages = [
        #{"role":"system", "content": "you are a helpful AI assistant. Answer concisely in short sentences"}# this system rule specifies sth that is not an assistant or user message, but that should be loaded in and understood by the LLM at the beginning of the conversation
        {"role":"user", "content": "Explain what an AI agent is in one sentence."},
        #{"role":"assiatant", "content": "An AI agent is simply an LLM connected to tools in a loop"}# assistant is tracking what the LLM has said previously, we need to keep track of that and pass it back to the LLM.
        ]
        #LLMs don't have any memory, so we bulid our own tools to store memory and retrieve it for LLM
)

print(response.choices[0].message.content)

##### local ai

In [4]:
%pip install ollama
import ollama

response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": "Explain what an AI agent is in one sentence."
        }
    ]
)

print(response.message.content)

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.



   ---------- ----------------------------- 1/4 [httpcore]
   ---------- ----------------------------- 1/4 [httpcore]
   -------------------- ------------------- 2/4 [httpx]
   -------------------- ------------------- 2/4 [httpx]
   ---------------------------------------- 4/4 [ollama]

Note: you may need to restart the kernel to use updated packages.
An AI agent is a software program that uses artificial intelligence (AI) and machine learning algorithms to interact with and make decisions within a specific environment or system, often simulating human-like decision-making and behavior.


In [5]:

response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a student in a university social network. "
                "Make decisions based on your beliefs and relationships."
            )
        },
        {
            "role": "user",
            "content": (
                "Your close friend tells you a rumour about another student. "
                "You are uncertain whether it is true. "
                "Would you share it with another friend? Explain your decision briefly."
            )
        }
    ]
)

print(response.message.content)

I wouldn't share it with another friend. I would consider the potential consequences of spreading a rumor, even if it's just a casual one, and decide to keep it to myself for now. I want to verify the accuracy of the rumor before sharing it with anyone, to avoid causing unnecessary harm or reputational damage to the student involved.


### step2 Keep track of the conversations that occurred. 

**There is no memory on the server**. The model forgets you the moment it answers. 

**We keep a list** and resend the **entire conversation** with every single request. 

The list is a **context window**. When it gets huge, real agents summarize it out. That's what **compaction** means. 

 1. Context window means how much space, or how many messages, it can effectively store. 



In [ ]:
messages = [] #This is the list that we are going to use. It populates and keeps adding to it to keep track of the conversation history. 

while True: #Just let the conversation go back and forth until you choose to exit it 
    user_input = input("You: ")
    if user_input.strip().lower() in ("exit","quit"):
        break
    messages.append({"role":"user", "content": user_input})

    response = ollama.chat(
        model = "llama3.2",
        messages = messages
        )

    reply = response.message.content
    messages.append({"role":"user", "content": reply})
    print("Bot:", reply)


It looks like you just sent a single letter "h". Is there something I can help you with?


### step3 Tools calling -from text generation to action selection and execution.

Step 1 — Define tools  
The programmer defines functions and their schemas(descriptions of functions).

Step 2 — Present available tools  
The model receives descriptions of the available actions and their parameters.Typically, this will come in as part of the system prompt. 

Step 3 — Choose a task  
Given its current situation, the model decides whether to request a tool call and supplies the arguments.

Step 4 — Execute the task  
**Harness**(like Langchain) maps the requested tool name to the actual function and executes it(python file).

Step 5 — Return the result  
The result is provided to the model, which can use it to decide what to do next.


In [ ]:

import ollama

def read_file(path):
    with open(path,"r", encoding = "UTF-8" )as f:
              return f.read()


Tool_schemas = [
       {
              "type": "function",
              "function":{
                     "name": "read_file",
                     "description": "Read a text file and retain its content. ",
                     "parameters":{
                            "type":"object",
                            "properties":{
                                   "path":{
                                          "tyoe": "string", 
                                          "description": "Path of the file to read."},
                             },
                            "required":["path"]
                        }
                }
       }
]#A schema is essentially a breakdown of how the tool works, what the parameters are, and what it is going to return. 

messages = [
       {"role":"user", "content": "what is inside notes.txt? summarize it in one line"}
       ]


#specific implentation and execute
while True:
      response = ollama.chat(
             model = "llama3.2", 
             messages = messages, 
             tools = Tool_schemas, # ollama supports that pass python function to tools variable directly and atonomily develop schema, which need clear illustration in mentioned function
             #tools = [read_file]
             )
      message = response.message
      messages.append(message) # add everything in message, like content, role, tool_calls...
      
      
# No tool calls means the model is done and gave us a normal answer
      if not message.tool_calls:
       print(message.content)
       break
      
      for tool_call in message.tool_calls:
           tool_name = tool_call.function.name
           args = tool_call.function.arguments
           result = read_file(**args) 
           messages.append({
                "role": "tool",
                "tool_name": tool_name,
                "content": str(result)
           })


    

message component
![alt text](image.png)

### Final
Build this into a much better example that combines all of this and makes it a lot more realistic where we make a mini claude code

##### Four tools make it a coding agent

list_files

read_file

write_file

run_command

In [14]:
import ollama

print(ollama.list())

models=[Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 10, 10, 21, 44, 49, 718285, tzinfo=TzInfo(3600)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M'))]


In [ ]:
import os
import subprocess
import ollama
system_prompt = """
When the user asks you to create a program:
1. Write the complete source code to a file using the write_file tool.
2. Save files inside the requested project directory.
3. Verify the file exists using list_files before running it.
4. Do not claim that a file was created unless write_file succeeded.
5. If you need to run the program, use run_command and ask for approval.
"""

def list_files(path = "."):
    entries = []
    for entry in os.scandir(path):
        entries.append(entry.name + ("/" if entry.is_dir() else ""))
    return "\n".join(sorted(entries)) or "(empty directory)"

def read_file(path):
    with open(path, "r", encoding = "utf-8") as f:
        return f.read()

def write_file(path, content):
    with open(path, "w", encoding = "utf-8") as f:
        f.write(content)
    return f"saved {path} ({len(content)} characters)"

def run_command(command): 
    answer = input (f"Run '{command}'? [y/N]")
    if answer.strip().lower() != "y":
        return "The user declined to run this command."
    result = subprocess.run(
        command, shell = True, capture_output = True, text = True, timeout = 120
    )
    output = (result.stdout + result.stderr).strip()
    return output or f"(no output, exit code{result.returncode})"

Tools = {
     "list_files" : list_files,
     "read_file" : read_file,
     "write_file" : write_file,
     "run_command" : run_command,
     }

Tool_schemas = [{
    "type": "function",
    "function":{
        "name": "list_files",
        "description": "list the file in a directory. Folder end with /.",
        "parameters":{
            "type": "object",
            "propeties":{
                "path":{"type": "string", "description" : "Diectory to list,e.g.'.'"}
            },
            "required": ["path"]
        }
    }
},{
    "type": "function",
    "function":{
        "name": "read_file",
        "description": "Read a text file and return its content.",
        "parameters":{
            "type": "object",
            "propeties":{
                "path":{"type": "string", "description" : "Path of the file to read."}
            },
             "required": ["path"]
        }
    }
},{
    "type": "function",
    "function":{
        "name": "write_file",
        "description": "Create or overwrite a text file with the given content.",
        "parameters":{
            "type": "object",
            "propeties":{
                "path":{"type": "string", "description" : "Path of the file to write"},
                "content":{"type": "string", "description" : "Full contents of the file"}
            },
            "required": ["path","content"]
        }
    }
},{
    "type": "function",
    "function":{
        "name": "run_command",
        "description": "Run a shell command and return its output. The user approves it first.",
        "parameters":{
            "type": "object",
            "propeties":{
                "command":{"type": "string", "description" : "The shell command to run"}
            },
            "required": ["command"]
        }
    }   
}]


#Then, we are going to have a function that will actually run the tool. 
def run_tool(tool_call):
    name = tool_call.function.name
    args = tool_call.function.arguments
    print(f"tool: {name}({args})")
    try:
        return str(Tools[name](**args))
    except Exception as error:
        return f"Error: {error}"

#def a function(more reusable) to Represent one run of our AI agent. (step 3)
def run_agent(messages):
    while True:#for one prompt
        response = ollama.chat(
             model = "llama3.2", 
             messages = messages, 
             tools = Tool_schemas,)
        message = response.message
        messages.append(message)

        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            result = run_tool(tool_call)
            messages.append({
                "role": "tool",
                "tool_name": tool_name,
                "content": result,
            })

def main():
    messages = [{"role": "system", "content": system_prompt}]
    print("Mini agent ready. Type exit to quit.")
    while True:
        user_input = input("\nYou:")
        if user_input.strip().lower() in ("exit","quit"):
            break
        messages.append({"role": "user", "content": user_input})
        reply = run_agent(messages)
        print(f"\nAgent: {reply}")

if __name__ == "__main__":
    main() #main() is main programming, other functions are called by main()
        





Mini agent ready. Type exit to quit.


In [17]:
import os

print(os.listdir("Snake"))

[]


In [1]:
#claide

In [ ]:
import os
import subprocess
import ollama

MODEL = "qwen2.5:7b"       # 先执行: ollama pull qwen2.5:7b (也可换成 llama3.1:8b)
MAX_STEPS = 10             # 单次提问内,模型最多连续调用工具的轮数
MAX_OUTPUT_CHARS = 5000    # 工具返回内容的最大长度,防止撑爆上下文
WORKSPACE = os.path.abspath(".")  # 所有文件操作限制在这个目录内

system_prompt = f"""
You are a coding assistant. Your working directory is: {WORKSPACE}
Rules:
- Always use RELATIVE paths such as "snake_game/main.py". Never use absolute paths.
- write_file needs BOTH "path" (a file, not a directory) and "content" (the full source code only, no chat text).
- Directories are created automatically by write_file, so never try to create a directory by itself.
- After writing, verify the file exists with list_files.
- Never say a file was created unless write_file returned "saved ...".
- To run a program, use run_command (the user will be asked for approval).
- Call tools through the real tool-calling mechanism, never by writing JSON as plain text.
"""


# ---------- 工具函数 ----------

def truncate(text, limit=MAX_OUTPUT_CHARS):
    if len(text) > limit:
        return text[:limit] + f"\n...(truncated, {len(text) - limit} more characters)"
    return text


def safe_path(path):
    """把路径解析到 WORKSPACE 内,禁止跳出工作目录。"""
    full = os.path.abspath(os.path.join(WORKSPACE, path))
    if os.path.commonpath([WORKSPACE, full]) != WORKSPACE:
        raise ValueError(f"path '{path}' is outside the workspace; use a relative path")
    return full


def list_files(path="."):
    full = safe_path(path)
    with os.scandir(full) as it:
        entries = [e.name + ("/" if e.is_dir() else "") for e in it]
    return "\n".join(sorted(entries)) or "(empty directory)"


def read_file(path):
    with open(safe_path(path), "r", encoding="utf-8") as f:
        return truncate(f.read())


def write_file(path, content):
    full = safe_path(path)
    if os.path.isdir(full):
        raise IsADirectoryError(f"'{path}' is a directory; give a file path such as '{path}/main.py'")
    os.makedirs(os.path.dirname(full), exist_ok=True)  # 目录不存在时自动创建
    with open(full, "w", encoding="utf-8") as f:
        f.write(content)
    return f"saved {path} ({len(content)} characters)"


def run_command(command):
    answer = input(f"Run '{command}'? [y/N] ")
    if answer.strip().lower() != "y":
        return "The user declined to run this command."
    try:
        result = subprocess.run(
            command, shell=True, capture_output=True, text=True,
            timeout=120, cwd=WORKSPACE,
        )
    except subprocess.TimeoutExpired:
        return "Error: command timed out after 120 seconds."
    output = (result.stdout + result.stderr).strip()
    return truncate(output) if output else f"(no output, exit code {result.returncode})"


TOOLS = {
    "list_files": list_files,
    "read_file": read_file,
    "write_file": write_file,
    "run_command": run_command,
}

TOOL_SCHEMAS = [{
    "type": "function",
    "function": {
        "name": "list_files",
        "description": "List the files in a directory. Folders end with /.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Directory to list, e.g. '.'"}
            },
            "required": []
        }
    }
}, {
    "type": "function",
    "function": {
        "name": "read_file",
        "description": "Read a text file and return its content.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Path of the file to read."}
            },
            "required": ["path"]
        }
    }
}, {
    "type": "function",
    "function": {
        "name": "write_file",
        "description": "Create or overwrite a text file with the given content. Parent directories are created automatically.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Path of the file to write, e.g. 'snake_game/main.py'."},
                "content": {"type": "string", "description": "Full contents of the file."}
            },
            "required": ["path", "content"]
        }
    }
}, {
    "type": "function",
    "function": {
        "name": "run_command",
        "description": "Run a shell command and return its output. The user approves it first.",
        "parameters": {
            "type": "object",
            "properties": {
                "command": {"type": "string", "description": "The shell command to run."}
            },
            "required": ["command"]
        }
    }
}]

# 每个工具的必填参数,用于报错时提示模型
REQUIRED_PARAMS = {
    s["function"]["name"]: s["function"]["parameters"]["required"]
    for s in TOOL_SCHEMAS
}


# ---------- 执行工具 ----------

def run_tool(tool_call):
    name = tool_call.function.name
    args = tool_call.function.arguments or {}
    print(f"tool: {name}({args})")
    if name not in TOOLS:
        return f"Error: unknown tool '{name}'. Available tools: {', '.join(TOOLS)}"
    try:
        return str(TOOLS[name](**args))
    except TypeError as error:
        return (f"Error: wrong arguments for '{name}': {error}. "
                f"Call the tool again with all required parameters: {REQUIRED_PARAMS[name]}")
    except Exception as error:
        return f"Error: {type(error).__name__}: {error}"


# ---------- 一次 agent 运行(处理一条用户输入) ----------

def run_agent(messages):
    for _ in range(MAX_STEPS):
        try:
            response = ollama.chat(
                model=MODEL,
                messages=messages,
                tools=TOOL_SCHEMAS,
            )
        except Exception as error:
            return (f"[Failed to call Ollama: {error}]\n"
                    f"Check that Ollama is running and the model is installed "
                    f"(ollama pull {MODEL}).")

        message = response.message
        messages.append(message)

        if not message.tool_calls:
            text = (message.content or "").strip()
            # 兜底:模型把工具调用当成普通文本输出了
            if text.startswith("{") and '"name"' in text:
                messages.append({
                    "role": "user",
                    "content": "You wrote a tool call as plain text, so nothing was executed. "
                               "Use the real tool-calling mechanism instead.",
                })
                continue
            return message.content

        for tool_call in message.tool_calls:
            result = run_tool(tool_call)
            messages.append({
                "role": "tool",
                "tool_name": tool_call.function.name,
                "content": result,
            })

    return f"[Stopped: reached the limit of {MAX_STEPS} tool-call rounds.]"


# ---------- 主程序 ----------

def main():
    messages = [{"role": "system", "content": system_prompt}]
    print(f"Mini agent ready (model: {MODEL}). Type exit to quit.")
    while True:
        try:
            user_input = input("\nYou: ")
        except (EOFError, KeyboardInterrupt):
            print("\nBye.")
            break
        if user_input.strip().lower() in ("exit", "quit"):
            break
        if not user_input.strip():
            continue
        messages.append({"role": "user", "content": user_input})
        reply = run_agent(messages)
        print(f"\nAgent: {reply}")


if __name__ == "__main__":
    main()

Mini agent ready (model: qwen2.5:7b). Type exit to quit.

Agent: Sure, let's start by creating the `Snake` folder and then we'll write the Python code for the snake game inside it.

First, we need to create the `Snake` folder. Do you want to proceed with that?
tool: write_file({'path': 'Snake/', 'content': ''})
tool: write_file({'path': 'Snake/main.py', 'content': "# This is a basic implementation of the Snake game in Python.\n# For simplicity, we will use basic console-based graphics.\n\nimport random\nimport os\n\n# Function to clear the screen\ndef clear():\n    os.system('cls' if os.name == 'nt' else 'clear')\n\ndef game():\n    width, height = 20, 10\n    snake = [(5, 5)]\n    food = (random.randint(0, width-1), random.randint(0, height-1))\n    direction = 'RIGHT'\n    score = 0\n\n    while True:\n        clear()\n        print('Score:', score, '\\n')\n        for y in range(height):\n            for x in range(width):\n                if (x, y) == food:\n                    pri